# Kaggle Workshop

This is the template for the House Price competition workshop. You can try to follow these proposed tasks during the workshop. There is also a [Workshop Walkthrough](https://www.kaggle.com/pvlima/workshop-walkthrough) kernel with simple baseline implementation that you can check as reference.

Have fun! 


In [5]:
%matplotlib inline
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import is_numeric_dtype

sns.set()

# Load data

In [6]:
rawtrain = pd.read_csv('../input/train.csv')
rawtest = pd.read_csv('../input/test.csv')

In [7]:
print('Train shape:', rawtrain.shape)
print('Test shape:', rawtest.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


These are the types of the columns in the dataset. `np.object` are string values for the categorical features.

In [8]:
rawtrain.dtypes.value_counts()

object     43
int64      35
float64     3
dtype: int64

# First model with selected features

To make it a bit easier to do the first steps with the dataset you can use the following list with the 20 most important features (this list is the result of running a gradient boosting model and selecting the most important features). 

In [9]:
selected = ['GrLivArea',
 'LotArea',
 'BsmtUnfSF',
 '1stFlrSF',
 'TotalBsmtSF',
 'GarageArea',
 'BsmtFinSF1',
 'LotFrontage',
 'YearBuilt',
 'Neighborhood',
 'GarageYrBlt',
 'OpenPorchSF',
 'YearRemodAdd',
 'WoodDeckSF',
 'MoSold',
 '2ndFlrSF',
 'OverallCond',
 'Exterior1st',
 'YrSold',
 'OverallQual']

Or select everything if you prefer

In [10]:
#features = [c for c in test.columns if c not in ['Id']]

This code builds a single dataframe with both `train` and `test` datasets and a new column to separate both. This can be useful when doing transformations that would need to be applied both in `train` and `test`. If you keep this approach you can use the checking code that is provided.

In [11]:
train = rawtrain[selected].copy()
train['is_train'] = 1
train['SalePrice'] = rawtrain['SalePrice'].values
train['Id'] = rawtrain['Id'].values

test = rawtest[selected].copy()
test['is_train'] = 0
test['SalePrice'] = 1  #dummy value
test['Id'] = rawtest['Id'].values

full = pd.concat([train, test])

not_features = ['Id', 'SalePrice', 'is_train']
features = [c for c in train.columns if c not in not_features]

# Check target distribution

The competition metric is based on log transformed values. That is already an hint that log transform maybe useful to make the target distribution behave more like a normal distribution.

Try to plot the distribution of `SalePrice`.

And apply the log transformation to `SalePrice` in the dataset.

# Check missing values

Do some analysis to identify the missing values. There is a proposed `summary` function that can be used to check missing values for the different dtypes (`np.object`, `np.float64`, `np.int64`). 

In [12]:
def summary(df, dtype):
    data = []
    for c in df.select_dtypes([dtype]).columns:
        data.append({'name': c, 'unique': df[c].nunique(), 
                     'nulls': df[c].isnull().sum(),
                     'samples': df[c].unique()[:20] })
    return pd.DataFrame(data)

Now do something to replace the missing values. The best is to analyse case by case. A quick lazy approach can be to use a new label missing categoricals and zero for missing numerical.

Some code to check there are no missing values in the dataset (uncomment)

In [13]:
#for c in full.columns:
#    assert full[c].isnull().sum() == 0, f'There are still missing values in {c}'

# Encode categorical

Before creating the model the categorical features must be encoded into numberical values. There are many ways to do it, for example building a mapping dictionary and applying it with pandas. Or using `sklearn.preprocessing.LabelEncoder`.

Code to check that all columns are numeric

In [14]:
#for c in full.columns:
#    assert is_numeric_dtype(full[c]), f'Non-numeric column {c}'

# First model

Now try to build a first predictive model. One suggestion is to use gradient boosting that typically has strong results in this kind of tabular data (in `sklearn` there is a `GradientBoostingRegress` model). If you choose to do first a Linear regression don't forget to also one-hot encode the categorical values.

In [15]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics

Implementation of the competition metric (notice target is already log transformed)

In [16]:
def rmse(y_true, y_pred):
    return np.sqrt(metrics.mean_squared_error(y_true, y_pred))

Choose a validation strategy for your model. Simple approach is to take out a validation dataset from the train dataset (`sklearn.model_selection.train_test_split` can be used for this).

If the target was log transformed it needs to be exponentiated before creating the submission file. Then save the submission file.

In [17]:
#subm = pd.DataFrame(ypred, index=test['Id'], columns=['SalePrice'])
#subm.to_csv('submission.csv')

🎉 Great! Submission ready 💪 Now time to upload to Kaggle. 

# Remove Outliers

It is worth checking the data for outliers and try a model with some outliers removed. Suggested task is to plot a boxplot for each numerical variable. Then filter out some outliers.

# Feature engineering

Some ideas for new features:

- House age (considering the construction year and that this is a dataset from 2010)
- What season was the house sold (winter, summer, etc)
- Reduce the overal condition to 3 levels (good, average, bad)
- How long ago the house was sold
- Total area including first and second floor
- Total area including first floor, second floor and basement


# Blend 2 models

Now try to make 2 different models (for example GBM and ExtraTrees or RandomForest), combine the models (for example with a weighted average) and evaluate the performance in the validation set.

If the performance improved in you CV, do another submission on Kaggle to check the value on `test`.  

In [18]:
#subm = pd.DataFrame(blendtestpred, index=test['Id'], columns=['SalePrice'])
#subm.to_csv('submission_blend.csv')

Well Done! 🏆 Now just keep the momentum and go for the gold 🥇🥇🥇🥇🚀

# Extra: if you still have some time

Try the following:

- K-fold CV 
- Liner regression model 